In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
# MINIO CONFIGURATION
s3_host = "minio"
s3_url = f"http://{s3_host}:9000"
s3_key = "minio"
s3_secret = "SU2orange!"
s3_bucket = "labe"

5. Configure Spark to read from Minio labe bucket, then load syracuse-ny.csv into a DataFrame and register 
it as the table weather.  

In [4]:
# Spark init
spark = SparkSession.builder \
    .master("local") \
    .appName('jupyter-pyspark') \
    .config("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4")\
    .config("spark.hadoop.fs.s3a.endpoint", s3_url) \
    .config("spark.hadoop.fs.s3a.access.key", s3_key) \
    .config("spark.hadoop.fs.s3a.secret.key", s3_secret) \
    .config("spark.hadoop.fs.s3a.fast.upload", True) \
    .config("spark.hadoop.fs.s3a.path.style.access", True) \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider') \
    .getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print(f"the bucket is {s3_bucket}")

the bucket is labe


In [5]:
weather = spark.read\
    .option("header",True).option("inferSchema",True)\
    .csv("s3a://labe/syracuse-ny.csv")

weather.printSchema()

root
 |-- EST: timestamp (nullable = true)
 |-- Max TemperatureF: integer (nullable = true)
 |-- Mean TemperatureF: integer (nullable = true)
 |-- Min TemperatureF: integer (nullable = true)
 |-- Max Dew PointF: integer (nullable = true)
 |-- MeanDew PointF: integer (nullable = true)
 |-- Min DewpointF: integer (nullable = true)
 |-- Max Humidity: integer (nullable = true)
 |-- Mean Humidity: integer (nullable = true)
 |-- Min Humidity: integer (nullable = true)
 |-- Max Sea Level PressureIn: double (nullable = true)
 |-- Mean Sea Level PressureIn: double (nullable = true)
 |-- Min Sea Level PressureIn: double (nullable = true)
 |-- Max VisibilityMiles: integer (nullable = true)
 |-- Mean VisibilityMiles: integer (nullable = true)
 |-- Min VisibilityMiles: integer (nullable = true)
 |-- Max Wind SpeedMPH: integer (nullable = true)
 |-- Mean Wind SpeedMPH: integer (nullable = true)
 |-- Max Gust SpeedMPH: integer (nullable = true)
 |-- PrecipitationIn: string (nullable = true)
 |-- Clou

6. Rewrite Question 2 using pure Spark SQL and the weather temp view. NOTE: There will be some subtle 
differences with how you must write the code, so be sure to printSchema() so you can see what the 
columns are.  

In [7]:
spark.read\
    .option("header",True).option("inferSchema",True)\
    .csv("s3a://labe/syracuse-ny.csv")\
    .createOrReplaceTempView("weather")

spark.sql("select * from weather").printSchema()

root
 |-- EST: timestamp (nullable = true)
 |-- Max TemperatureF: integer (nullable = true)
 |-- Mean TemperatureF: integer (nullable = true)
 |-- Min TemperatureF: integer (nullable = true)
 |-- Max Dew PointF: integer (nullable = true)
 |-- MeanDew PointF: integer (nullable = true)
 |-- Min DewpointF: integer (nullable = true)
 |-- Max Humidity: integer (nullable = true)
 |-- Mean Humidity: integer (nullable = true)
 |-- Min Humidity: integer (nullable = true)
 |-- Max Sea Level PressureIn: double (nullable = true)
 |-- Mean Sea Level PressureIn: double (nullable = true)
 |-- Min Sea Level PressureIn: double (nullable = true)
 |-- Max VisibilityMiles: integer (nullable = true)
 |-- Mean VisibilityMiles: integer (nullable = true)
 |-- Min VisibilityMiles: integer (nullable = true)
 |-- Max Wind SpeedMPH: integer (nullable = true)
 |-- Mean Wind SpeedMPH: integer (nullable = true)
 |-- Max Gust SpeedMPH: integer (nullable = true)
 |-- PrecipitationIn: string (nullable = true)
 |-- Clou

In [8]:
query = '''
with source as  (
    select 
        cast(split(EST,'-')[0] as int) as year,
        cast(split(EST,'-')[1] as int) as month,
        `Min TemperatureF` as mintemp,
        `Max TemperatureF` as maxtemp
    from weather
)
select 
    year, month, avg(mintemp) as avgmin, avg(maxtemp) as avgmax
    from source 
    group by year, month
    order by year, month

'''
spark.sql(query).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [year#237 ASC NULLS FIRST, month#238 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(year#237 ASC NULLS FIRST, month#238 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=83]
      +- HashAggregate(keys=[year#237, month#238], functions=[avg(mintemp#239), avg(maxtemp#240)])
         +- Exchange hashpartitioning(year#237, month#238, 200), ENSURE_REQUIREMENTS, [plan_id=80]
            +- HashAggregate(keys=[year#237, month#238], functions=[partial_avg(mintemp#239), partial_avg(maxtemp#240)])
               +- Project [cast(split(cast(EST#166 as string), -, -1)[0] as int) AS year#237, cast(split(cast(EST#166 as string), -, -1)[1] as int) AS month#238, Min TemperatureF#169 AS mintemp#239, Max TemperatureF#167 AS maxtemp#240]
                  +- FileScan csv [EST#166,Max TemperatureF#167,Min TemperatureF#169] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[s3a://labe/syracuse-ny.

7. Save the output from the DataFrame in Question 6 to the temp view 
monthly_syracuse_weather_averages. Prove the view is there by querying it. 

In [9]:
query = '''
with source as  (
    select 
        cast(split(EST,'-')[0] as int) as year,
        cast(split(EST,'-')[1] as int) as month,
        `Min TemperatureF` as mintemp,
        `Max TemperatureF` as maxtemp
    from weather
)
select 
    year, month, avg(mintemp) as avgmin, avg(maxtemp) as avgmax
    from source 
    group by year, month
    order by year, month

'''
spark.sql(query).createOrReplaceTempView("monthly_syracuse_weather_averages")

spark.sql("show tables").show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|         |monthly_syracuse_...|       true|
|         |             weather|       true|
+---------+--------------------+-----------+



8. CHALLENGE YOURSELF! At the bottom of the work/content/E-Drill-Spark.ipynb file there is a section 
called Big Data to Small Data. Try to write a complete program that: 
a. Inputs a month 1–12 at run-time 
b. Displays a scatter plot of min/max average monthly temperatures, where year is on the X-axis 

In [11]:
month = input("Enter Month: ")
spark.sql(f"select * from monthly_syracuse_weather_averages where month = {month}").toPandas()

Enter Month:  9


,year,month,avgmin,avgmax
0,1997,9,51.066667,69.400000
1,1998,9,54.400000,73.800000
2,1999,9,54.800000,75.766667
3,2000,9,50.433333,71.200000
4,2001,9,52.166667,73.000000
5,2002,9,56.066667,78.200000
6,2003,9,53.600000,73.000000
7,2004,9,55.433333,74.966667
8,2005,9,54.500000,76.366667
9,2006,9,52.000000,69.500000


In [12]:
spark.sql(f"select max(year),max(month) from monthly_syracuse_weather_averages").show()

+---------+----------+
|max(year)|max(month)|
+---------+----------+
|     2015|        12|
+---------+----------+



In [14]:
from IPython.display import display, HTML
from ipywidgets import interact_manual
import matplotlib.pyplot as plt

display(HTML("<H1>Syracuse Weather</h1>") )
@interact_manual(Month=(1,12))
def doit(Month):
    df = spark.sql(f"select * from monthly_syracuse_weather_averages where month = {Month}").toPandas()
    display(df)
    df.set_index("year")
    plt.figure(figsize=(15,10))
    plt.scatter(df["year"],y=df["avgmin"], label='monthly avg min', marker='v')
    plt.scatter(df["year"],y=df["avgmax"], label='monthly avg max', marker='^')
    plt.legend(title=f"Temps for Month {Month}")
    plt.show()
    

interactive(children=(IntSlider(value=6, description='Month', max=12, min=1), Button(description='Run Interact…